# Deep Learning 1 &mdash; Assignment 6

Sixth assignment for the 2025 Deep Learning course (NWI-IMC070A) of the Radboud University.

-----

**Names:** Nele Haferkorn

**Group:** 13

-----

**Instructions:**
* Fill in your names and the name of your group.
* Answer the questions and complete the code where necessary.
* Keep your answers brief, one or two sentences is usually enough.
* Re-run the whole notebook before you submit your work.
* Save the notebook and submit the `.ipynb` file in Brightspace.

## Objectives

In this assignment you will
1. Construct your own PyTorch `DataSet`.
1. Train and modify a transformer network.
1. Learn how to use a model based on the PyTorch documentation.
1. Experiment with a translation dataset.

## Required software

If you haven't done so already, you will need to install the following additional libraries:
* `torch` for PyTorch,

All libraries can be installed with `pip install`.

In [ ]:
%matplotlib inline
import math
import time
from random import Random
from typing import (List, Optional)
import numpy as np
import torch
from torch import nn
from torch.utils.data import (IterableDataset, DataLoader)
import matplotlib.pyplot as plt
from IPython import display

# fix the seed, so outputs are exactly reproducible
torch.manual_seed(12345);

# Use the GPU if available
def detect_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")
device = detect_device()

## 6.1 Learning to calculate (5 points)

In this assignment we are going to train a neural network to do mathematics.
When communicating between humans, mathematics is expressed with words and formulas.
The simplest of these are formulas with a numeric answer. For example, we might ask what is `100+50`, to which the answer is `150`.

To teach a computer how to do this task, we are going to need a dataset.

Below is a function that generates a random formula. Study it, and see if you understand its parameters.

In [ ]:
def random_integer(length: int, signed: bool = True, rng: Random = Random()):
    max = int(math.pow(10, length))
    min = -max if signed else 0
    return rng.randint(min, max)

def random_formula(complexity: int, signed: bool = True, rng: Random = Random()):
    """
    Generate a random formula of the form "a+b" or "a-b".
    complexity is the maximum number of digits in the numbers.
    """
    a = random_integer(complexity, signed, rng)
    b = random_integer(complexity, False, rng)
    is_addition = not signed or rng.choice([False, True]) # not sure what this line does actually
    if is_addition:
        return (f"{a}+{b}", str(a + b))
    else:
        return (f"{a}-{b}", str(a - b))

In [ ]:
seed = 123456
random_formula(3, rng=Random(seed))

Note that the `rng` argument allows us to reproduce the same random numbers, which you can verify by running the code below multiple times. But if you change the seed to `None` then the random generator is initialized differently each time.

In [ ]:
def random_formulas(complexity, signed, count, seed):
    """
    Iterator that yields the given count of random formulas
    """
    rng = Random(seed)
    for i in range(count):
        yield random_formula(complexity, signed, rng=rng)

for q, a in random_formulas(3, True, 5, seed):
    print(f'{q} = {a}')

`yield` keyword in Python:  
- used to create **generators**, which are special types of iterators that allow values to be produced lazily, one at a time, instead of returning them all at once.

Not sure however

We are going to treat these expressions as sequences of tokens, where each character is a token. In addition we will need tokens to denote begin-of-sequence and end-of-sequence, as well as padding, for which we will use `'<bos>'`, `'<eos>'`, and `'<pad>'` respectively, as was done in the lecture.

### Creating a vocabulary

The vocabulary is the set of all possible tokens.

To store the vocabulary, we will use a helper class:

In [ ]:
class Vocab:
    """A vocabulary."""
    def __init__(self, tokens: List[str], specials: Optional[List[str]] = None):
        """
        Construct a vocabulary given a list of tokens, and an optional list of special tokens
        """
        self.tokens = tokens
        if specials is not None:
            self.tokens.extend(specials)
        self.tokens.sort()
        self.token_to_idx = {token: idx for idx, token in enumerate(self.tokens)}

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, token):
        return self.lookup_index(token)

    def lookup_index(self, token: str) -> int:
        return self.token_to_idx[token]

    def lookup_indices(self, tokens: List[str]) -> List[int]:
        return [self.lookup_index(token) for token in tokens]

    def lookup_token(self, index: int) -> str:
        return self.tokens[index]

    def lookup_tokens(self, indices: List[int]) -> List[str]:
        return [self.lookup_token(index) for index in indices]

For this dataset we know beforehand what the vocabulary will be, so we can easily define it by hand.

**(a) What are the tokens in this dataset? Complete the code below.<span style="float:right"> (1 point)</span>**

In [ ]:
# TODO: fill in all possible tokens
vocab = Vocab(['0','1','2','3','4','5','6','7','8','9', '-', '+'], specials=['<bos>','<eos>', '<pad>'])


We can print the vocabulary to double check that it makes sense:

In [ ]:
print('Vocabulary size:', len(vocab))
print('Vocabulary:', vocab.tokens)

We are now ready to tokenize and encode formula.

**(b) Complete the code below.<span style="float:right"> (1 point)</span>**

The function must return a list, and output must include encoded `<eos>` token

In [ ]:
def tokenize_and_encode(string: str, vocab=vocab) -> List[int]:
    # TODO: Tokenize the string and encode using the vocabulary.
    #       Include an end-of-string token (but not a begin-of-string token).
    tokens = list(string)
    tokens.append('<eos>')
    return vocab.lookup_indices(tokens)

So we get a string of characters, which is split at each character and turned into tokenized sequence.

Let's test it on a random formula:

In [ ]:
q, a = random_formula(3, rng=Random(seed))
print('The question', q, 'and answer', a)
print('are encoded as', tokenize_and_encode(q), 'and', tokenize_and_encode(a))

# Check tokenize_and_encode
assert ''.join(vocab.lookup_tokens(tokenize_and_encode(q))) == q + '<eos>'
assert len(tokenize_and_encode(q)) == len(q) + 1
assert tokenize_and_encode("1+2") == vocab.lookup_indices(['1','+','2','<eos>'])

### Padding and trimming

Next, to be able to work with a whole dataset of these encoded sequences, they all need to be the same length.

**(c) Implement the function below that pads or trims the encoded token sequence as needed.<span style="float:right"> (1 point)</span>**

Hint: see [d2l section 10.5.3](http://d2l.ai/chapter_recurrent-modern/machine-translation-and-dataset.html#loading-sequences-of-fixed-length) for a very similar function.

In [ ]:
## Official Solution
def pad_or_trim(tokens: List[int], target_length: int, vocab=vocab):
    """Pad (or trim) a list of tokens so that it has a given target length."""
    ### BEGIN ANSWER
    padding = [vocab['<pad>']] * max(0, target_length - len(tokens))
    return tokens[:target_length] + padding
    ### END ANSWER

In [ ]:
def pad_or_trim(tokens: List[int], target_length: int, vocab=vocab):
    """Pad (or trim) a list of tokens so that it has a given target length."""
    # TODO return a padded or trimmed sequence
    # perform trimming
    if len(tokens) > target_length:
        return tokens[:target_length]
    elif len(tokens) < target_length:
        # perform padding
        return tokens + [vocab['<pad>']] * (target_length - len(tokens))
    else:
      return tokens

In [ ]:
# pad or trim q to get a sequence of 10 tokens
pad_or_trim(tokenize_and_encode(q), 10)

In [ ]:
# Check pad_or_trim
assert len(pad_or_trim([1,2,3,4,5],10)) == 10
assert len(pad_or_trim(list(range(20)),10)) == 10
assert vocab.lookup_tokens(pad_or_trim([1,2,3,4,5],10)[5:]) == ['<pad>','<pad>','<pad>','<pad>','<pad>'], \
       f"Incorrect padding tokens, found {vocab.to_tokens(pad_or_trim([1,2,3,4,5],10)[5:])}"

### Translating tokens

We can use `vocab.lookup_tokens` to convert the encoded token sequence back to something more readable:

In [ ]:
vocab.lookup_tokens(pad_or_trim(tokenize_and_encode(q), 10))

For convenience, we define the `decode_tokens` function to convert entire lists or tensors:

In [ ]:
def decode_tokens(t, vocab=vocab):
    # convert a list, tensor, or array of encoded tokens
    if isinstance(t, torch.Tensor):
        t = t.detach().cpu()
    else:
        t = torch.tensor(t)
    return np.asarray(vocab.lookup_tokens(list(t.flatten()))).reshape(*t.shape)

# convert all tokens at once
print(decode_tokens([pad_or_trim(tokenize_and_encode('513+1323'), 10),
                     pad_or_trim(tokenize_and_encode('412+42'), 10)]))

### Creating a dataset

The most convenient way to use a data generating function for training a neural network is to wrap it in a PyTorch `Dataset`. In this case, we will use an [IterableDataset](https://pytorch.org/docs/stable/data.html#torch.utils.data.IterableDataset), which can be used as an iterator to walk over the samples in the dataset.

**(d) Complete the code below.<span style="float:right"> (1 point)</span>**

In [ ]:
class FormulaDataset(IterableDataset):
    def __init__(self, complexity, signed, count, seed=None, vocab=vocab):
        self.seed = seed
        self.complexity = complexity
        self.signed = signed
        self.count = count
        self.vocab = vocab
        self.max_question_length = 2 * complexity + 3
        self.max_answer_length = complexity + 3

    def __len__(self):
        return self.count


    def __iter__(self):
        for q, a in random_formulas(self.complexity,self.signed, self.count, self.seed):

            # Tokenize and encode
            question_encoded = tokenize_and_encode(q, self.vocab)
            answer_encoded = tokenize_and_encode(a, self.vocab)

            # perform padding or trimming
            question_padded = pad_or_trim(question_encoded, self.max_question_length, self.vocab)
            answer_padded = pad_or_trim(answer_encoded, self.max_answer_length, self.vocab)

            # create torch tensors
            yield torch.tensor(question_padded), torch.tensor(answer_padded)

In [ ]:
## Official Solution - Correct Implementation
class FormulaDataset(IterableDataset):
    def __init__(self, complexity, signed, count, seed=None, vocab=vocab):
        self.seed = seed
        self.complexity = complexity
        self.signed = signed
        self.count = count
        self.vocab = vocab
        self.max_question_length = 2 * complexity + 3
        self.max_answer_length = complexity + 3

    def __len__(self):
        return self.count

    ### BEGIN ANSWER
    def __iter__(self):
        # Restart at the same seed in every epoch
        rng = Random(self.seed)
        for i in range(self.count):
            q, a = random_formula(self.complexity, self.signed, rng=rng)
            q = torch.tensor(pad_or_trim(tokenize_and_encode(q, self.vocab),
                                         self.max_question_length, self.vocab))
            a = torch.tensor(pad_or_trim(tokenize_and_encode(a, self.vocab),
                                         self.max_answer_length, self.vocab))
            yield q, a
    ### END ANSWER

**(e) Define a training set with 10000 formulas and a validation set with 5000 formulas, both with complexity 3.<span style="float:right"> (1 point)</span>**

Note: make sure that the training and validation set are different.

In [ ]:
complexity = 3
signed = True
# TODO: Your code here.
train_data = FormulaDataset(complexity, signed, 10000, seed=123)
val_data   = FormulaDataset(complexity, signed, 5000, seed=456)

As usual, we wrap each dataset in a `DataLoader` to create minibatches.

In [ ]:
# Define data loaders
batch_size = 125
data_loaders = {
    'train': torch.utils.data.DataLoader(train_data, batch_size=batch_size),
    'val':   torch.utils.data.DataLoader(val_data, batch_size=batch_size),
}

In [ ]:
# The code below checks that the datasets are defined correctly
train_loader = data_loaders['train']
val_loader = data_loaders['val']

from typing import Tuple
from typing_extensions import assert_type
for (name, loader), expected_size in zip(data_loaders.items(), [10000,5000]):
    first_batch = next(iter(loader))
    assert len(first_batch) == 2, \
           f"The {name} dataset should yield (question, answer) pairs when iterated over."
    assert torch.is_tensor(first_batch[0]), \
           f"The questions in the {name} dataset should be torch.tensors"
    assert tuple(first_batch[0].shape) == (batch_size, 2*complexity+3), \
           f"The questions in the {name} dataset should be of size (batch_size, max_question_length), i.e. {batch_size,2*complexity+3}, found {tuple(first_batch[0].shape)}"
    assert first_batch[0].dtype in [torch.int32,torch.int64], \
           f"The questions in the {name} dataset should be encoded as integers, found {first_batch[0].dtype}"
    assert torch.equal(next(iter(loader))[0], next(iter(loader))[0]), \
           f"The {name} dataset should be deterministic, it should produce the same data each time"
    assert all([len(batch[0]) == batch_size for batch in iter(loader)]), \
           f"Batches should all have the right size. Perhaps the batch size does not evenly divide the dataset size?"
    assert sum([len(batch[0]) for batch in iter(loader)]) == expected_size, \
           f"{name} dataset does not have the right size, expected {expected_size}, found {sum([len(batch[0]) for batch in iter(loader)])}."
    assert not torch.equal(next(iter(train_loader))[0], next(iter(val_loader))[0]), \
       "The training data and validation data should not be the same"

## 6.2 Transformer inputs (10 points)

There is a detailed description of the transformer model in Bishop chapter 12, and in [d2l chapter 11](http://d2l.ai/chapter_attention-mechanisms-and-transformers/index.html). We will not use the code from the d2l book, and instead use [PyTorch's built-in Transformer layers](https://pytorch.org/docs/stable/nn.html#transformer-layers).

However, some details we still need to implement ourselves.

### Autoregressive inputs

We will be training the decoder of the transformer as an autoregressive model.

The code below takes a batch of data from the training set, and it generates a shifted version of the target values.

In [ ]:
def shift_targets(y, bos_token=vocab['<bos>']):
    """
    Shift a sequence of tokens by 1 position, and add `bos_token` at the start.
    """
    bos = torch.tensor(bos_token, dtype=y.dtype, device=y.device).expand(y.shape[0], 1)
    y_prev = torch.cat((bos, y[:, :-1]), axis=1)
    return y_prev

In [ ]:
x, y = next(iter(train_loader))
y_prev = shift_targets(y)

# print the first five samples
print(decode_tokens(y)[:5])
print(decode_tokens(y_prev)[:5])

**(a) Look at the values for the example above. What is `y_prev` used for during training of a transformer model?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
To ensure causality.
`y_prev` will be used to make sure that the prediction of the transformer model only depends on past elements in the sequence.
This is called masked self-attention. The target values are shifted by one position so that the prediction for position p relies only on outputs p-1.

**OFFICIAL SOLUTION:**  
`y_prev` is the previous predicted/output toke. During training, each token in the output can 'see' the previous output tokens. We shift the input and output, so that `y[i]` is based on `y_prev[i] = y[i-1]`.

**(b) Why do some rows of `y_prev` end in `'<eos>'`, but not all? Is this a problem?<span style="float:right"> (1 point)</span>**

**ANSWER:**   
In addition the input sequences are of different lengtsh and are padded.
Thus, for shorter sequences, y_prev may end with <pad> instead of <eos>.
For long sequences, the <eos> may appear before padding, so in some rows it appears at the end.

**OFFICIAL SOLUTION:**  
`<eos>` only appears in `y_prev` if there was a padding token in y. This is not a problem. The padding tokens are ignored in the loss function. So cases where the transformer is asked to predict the next token after `<eos>` don't contribute to the training.

### Masks

Training a transformer uses masked self-attention, so we need some masks. Here are two functions that make these masks.

In [ ]:
def generate_square_subsequent_mask(size, device=device):
    """
    Mask that indicates that tokens at a position are not allowed to attend to
    tokens in subsequent positions.
    """
    mask = (torch.tril(torch.ones((size, size), device=device))) == 0
    return mask

def generate_padding_mask(tokens, padding_token):
    """
    Mask that indicates which tokens should be ignored because they are padding.
    """
    if not isinstance(tokens, torch.Tensor):
        tokens = torch.tensor(tokens)
    return tokens == padding_token

**(c) Generate a padding mask for a random encoded token string.<span style="float:right"> (1 point)</span>**

Hint: make sure that `tokens` is a torch.tensor.

In [ ]:
q, a = random_formula(3, rng=Random(seed))

tokens = torch.tensor(tokenize_and_encode(q, vocab=vocab))
padding_mask = generate_padding_mask(tokens, padding_token = vocab['<pad>'])
print(tokens)
print(decode_tokens(tokens))
print(padding_mask)

In [ ]:
# More tests
assert list(generate_padding_mask(torch.tensor(pad_or_trim(tokenize_and_encode("1+1"), 8)), vocab['<pad>'])) == [False]*4 + [True]*4, "Something is wrong with generate_padding_mask"

In [ ]:
## Correct Solution
# i think I forgot the input length argument in the tokenize function
q, a = random_formula(3, rng=Random(seed))
### BEGIN ANSWER
tokens = (pad_or_trim(tokenize_and_encode(q), 10))
padding_mask = generate_padding_mask(tokens, vocab['<pad>'])
### END ANSWER
print(tokens)
print(decode_tokens(tokens))
print(padding_mask)

**(d) How will this mask be used by a transformer?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
The padding mask will be used to ignore the <pad> tokens in the input sequences. While applying padding is necessary to make sure that the input sequences are the same length, they are not relevant for our actual task. Because they provide no meaningful information, the modek shouldn't pay attention to them. We use masks to tell the model which are the actual tokens and should be considered by the model and which are the tokens which should be ignores.

**OFFICIAL SOLUTION:**  
The padding mask is used in the transformer to mask out parts of the attention mechanism: other tokens are not allowed to attend to padding tokens.In this way the output for normal tokens doesn't depend on the padding token at all. The same mask is also used for the loss function: the cross-entropy for padding tokens are not in the final loss.

The code below illustrates what the output of `generate_square_subsequent_mask` looks like.

In [ ]:
square_subsequent_mask = generate_square_subsequent_mask(y.shape[1])

print(square_subsequent_mask.shape)
print(square_subsequent_mask)

**(e) How and why should this mask be used? State your answer in terms of `x`,  `y` and/or `y_prev`.<span style="float:right"> (1 point)</span>**

**ANSWER:**  
Because we don't want future tokens to influence the current prediction, so we hide them with a mask. In this way, the transformer only attends to previous tokens.
The first row represents the mask when focusing on the first token, meaning only the first token is considered. This indicates that the computation of attention is only performed for tokens preceding the current focus token.
Stated in terms of y and y_prev, y at time t must depend only on y_prev, not on future values of y. The mask is only applied to y (the token sequence, but not x (the input sequence).

**OFFICIAL SOLUTION:**  
This mask indicates to which tokens in `y_prev` every predicted token in y is not allowed to attend. The True values above the diagonal mean that `y[i]` can only depend on `y_prev[j] for j <= i`. This means that the transformer can not look at 'later' tokens. This is called training with masked self attention.

**(f) Give an example where it could make sense to use a different mask in a transformer network, instead of the `square_subsequent_mask`?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
For problems where it is important to take both the left and right sides context into account. Because the square subsequent mask creates a triangular masking matrix and only works if the autoregression assumption is not violated.


**OFFICIAL SOLUTION**:  
The self attention mask in the question x is an all False matrix (or no mask is used), meaning that all attention is allowed. This makes sense because the whole input is always available at once.
The cross attention mask is all False, so all attention from output onto input is allowed.
When the output is not a sequence, then a different mask can be used. Think of a tree or a set.
For a right-to-left sequence the flipped mask could be used.

### Embedding

Our discrete vocabulary is not suitable as the input for a transformer. We need an embedding function to map our input vocabulary to a continuous, high-dimensional space.

We will use the `torch.nn.Embedding` class to for this. As you can read in the [documentation](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html#torch.nn.Embedding), this class maps each token in our vocabulary to a specific point in embedding space, its embedding vector. We will use this embedding vector as the input features for the next layer of our model.

The parameters of the embedding are trainable: the embedding vector of each token is optimized along with the rest of the network.

**(g) Define an embedding that maps our vocabulary to a 5-dimensional space.<span style="float:right"> (1 point)</span>**

In [ ]:
# TODO: Your code here.
embedding = torch.nn.Embedding(num_embeddings=len(vocab), embedding_dim=5)
print(embedding)

Let's apply the embedding to some sequences from our training set.

In [ ]:
# take the first batch
x, y = next(iter(train_loader))
# take three samples
x = x[:3]
# print the shapes
print(x)
print(embedding(x))
print(x.shape)
print(embedding(x).shape)

**(h) Explain the output shape.<span style="float:right"> (1 point)</span>**

**ANSWER:**    
Our input tokens are now projected to a five-dimensional space. The original shape of x [3,9] means that there are 3 input sequences, each consisting of 9 tokens. After passing it through the embedding, each token gets replaced by a 5d vector.
Thats why we get a tensor of size (3,9,5).

The size of the embedding vectors, or the dimensionality of the embedding space, does not depend on the number of tokens in our vocabulary. We are free to choose an embedding size that fits our problem.

For example, let's try an embedding with 2 dimensions, and plot the initial embedding for the tokens in our vocabulary.

**(i) Create an embedding with 2 dimensions and plot the embedding for all tokens.<span style="float:right"> (no points)</span>**

In [ ]:
# TODO: Your code here.
embedding = torch.nn.Embedding(num_embeddings=len(vocab), embedding_dim=2)
# embed all tokens of our vocabulary
x = torch.arange(len(vocab))
emb = embedding(x).detach().cpu().numpy()

plt.scatter(emb[:, 0], emb[:, 1]);
for i, token in enumerate(vocab.tokens):
    plt.annotate(token, (emb[i,0]+0.04, emb[i,1]))

As always, we need to balance the complexity of our networks: a larger embedding will increase the number of parameters in our model, but increase the risk of overfitting.

**(j) Would this 2-dimensional embedding space be large enough for our problem?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
It looks like the 2-dimensional embedding space might just be large enough for our vocab size and task, since we get fairly distributed (and importantly also not overlapping) embedding vectors for the individual tokens.

**OFFICIAL SOLUTION**:  
Probably not in practice, especially because the embedding dimensionality is also used as the size of vectors in each intermediate layer of the transformer model.

But you can imagine encoding the token 0-9 in a line, and + and - as +1 and -1 in an orthogonal direction. Perhaps that is enough to do addition.

Instead of using an embedding, we could also use a simple one-hot encoding to map the words in the vocabulary to feature vectors. However, practical applications of natural language processing never do this. Why not?

**(k) Explain the practical advantage of embeddings over one-hot encoding.<span style="float:right"> (1 point)</span>**

**ANSWER:**  
The practical advantage of embeddings over one-hot encoding is that embeddings come with a learnable matrix of embedding weights, meaning that this approach is dynamic and they can encode relationships.
One-hot encoding on the other hand requires unique high-dimensional vector representations of tokens and they do not capture similarities between tokens.

Practically speaking, this increases computational (memory) efficiency.

## 6.3 `torch.nn.Transformer` (8 points)

<div style="float: left"><a href="https://cs.ru.nl/~gvtulder/vaswani-fig-1-highlight.png"><img src="https://cs.ru.nl/~gvtulder/vaswani-fig-1-highlight.png" width="300"></a></div>

We now have all required inputs for our transformer.

Consult the documentation for the [`torch.nn.Transformer`](https://pytorch.org/docs/stable/generated/torch.nn.Transformer.html) class of PyTorch. This class implements a full Transformer as described in ["Attention Is All You Need"](https://arxiv.org/pdf/1706.03762.pdf), the paper that introduced this architecture.

The `Transformer` class implements the main part of the of the Transformer architecture, shown highlighted in the image on the left (see also Fig. 1 in "Attention Is All You Need").

For a given input sequence, it applies one or more encoder layers, followed by one or more decoder layers, to compute an output sequence that we can then process further.

Because the `Transformer` class takes care of most of the complicated parts of the model, we can concentrate providing the inputs and outputs: the grayed-out areas in the image.

Check out the parameters for the `Transformer` class and the inputs and outputs of its `forward` function.
<br style="clear: both">

**(a) Which parameter of the `Transformer` class should we base on our embedding?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
The parameter `d_model`, because it defines the dimensionality of each tokens embedding vector.

`d_model` = number of expected features in the encoder/decoder inputs.

**(b) Given fixed input and output dimensions, which parameters of the `Transformer` can we use to change the number of parameters of our network?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
We can adapt `dim_feedforward` (so the dimensionality of the feedforward network model), and also how many encoding / decoding layers we want to include in our architecture (`num_encoder_layers`, `num_decoder_layers`). With more layers, we get more parameters, since they are each associated with weight matrices. And finally, we could also increase / decrease the number of attention heads (`n_heads`).

**OFFICIAL SOLUTION**:  
num_encoder_layers, num_decoder_layers, and dim_feedforward. The first two control the depth of the network, the controls the complexity of the MLP layer.

Arguably dropout controls the complexity as well by doing regularization.

The dimensionality of the key, query, and value in torch.MultiHeadAttention is d_model//nhead, so the number of parameters stays the same if you change nhead. If you increase the number of heads, each head gets smaller vectors, so the overall complexity stays about the same

**(c) When can use the masks that we defined earlier when using the `forward` method of the `Transformer` class. Which masks should be used for which parameters?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
Masks should be applied to those components that perform attention operations.
The padding mask should be used on the input token sequence, so in the encoding layers or als decoding layers. While the square_subsequent_mask should be applied to the decoder part.
No masks should be applied to the feedforward layers.

**OFFICIAL SOLUTION**:
In the call to `forward` or `__call__` (that is, when applying the transformer):

`tgt_mask`: attention between tokens in the decoder, should use a `square_subsequent_mask`
`src_key_padding_mask`: padding tokens in the encoder self-attention, should use a `padding_mask`.
`memory_key_padding_mask`: padding tokens for cross attention from decoder to encoder, should use a `padding_mask`.
and `tgt_key_padding_mask`: padding tokens in the decoder, should use a `padding_mask`.


### Building a network

**(d) Complete the code for the TransformerNetwork.<span style="float:right"> (5 points)</span>**

Construct a network with the following architecture (see the image in the previous section for an overview):
1. An embedding layer that embeds the input tokens into a space of size `dim_hidden`.
2. A dropout layer (not shown in the image).
3. A [Transformer](https://pytorch.org/docs/stable/generated/torch.nn.Transformer.html) with the specified parameters (`dim_hidden`, `num_heads`, `num_layers`, `dim_feedforward`, and `dropout`).<br>Note: you will need to pass `batch_first=True`, to indicate that the first dimension runs over the batch and not over the sequence.
4. A final linear prediction layer that takes the output of the transformer to `dim_vocab` possible classes.

Don't worry about positional encoding for now, we will add that later.

The `forward` function should generate the appropriate masks and combine the layers defined in `__init__` to compute the output.

In [ ]:
class TransformerNetwork(torch.nn.Module):
    def __init__(self,
                 dim_vocab=len(vocab), padding_token=vocab['<pad>'],
                 num_layers=2, num_heads=4, dim_hidden=64, dim_feedforward=64,
                 dropout=0.01, positional_encoding=False):
        super().__init__()
        self.padding_token = padding_token
        self.embedding    = torch.nn.Embedding(num_embeddings=dim_vocab, embedding_dim=dim_hidden)
        self.dropout      = torch.nn.Dropout(p=dropout)
        self.transformer  = torch.nn.Transformer(
            d_model=dim_hidden,
            nhead=num_heads,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.predict      = torch.nn.Linear(dim_hidden, dim_vocab)
        self.positional_encoding_enabled = positional_encoding
        if positional_encoding:
            self.pos_encoding = torch.nn.Identity() # Fill this in later
        else:
            self.pos_encoding = torch.nn.Identity()

    def forward(self, src, tgt):
        # Apply embedding
        src_emb = self.dropout(self.embedding(src))
        tgt_emb = self.dropout(self.embedding(tgt))

        # create masks
        src_key_padding_mask = generate_padding_mask(src, self.padding_token)
        tgt_key_padding_mask = generate_padding_mask(tgt, self.padding_token)
        memory_key_padding_mask = src_key_padding_mask # Encoder output padding mask for decoder attention

        # square subsequent mask
        tgt_mask = generate_square_subsequent_mask(tgt.shape[1])

        # create output
        output = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            src_mask=None,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )

        # prediction
        output = self.predict(output)
        return output

In [ ]:
## Official solution
class TransformerNetwork(torch.nn.Module):
    def __init__(self,
                 dim_vocab=len(vocab), padding_token=vocab['<pad>'],
                 num_layers=2, num_heads=4, dim_hidden=64, dim_feedforward=64,
                 dropout=0.01, positional_encoding=False):
        super().__init__()
        self.padding_token = padding_token
        ### BEGIN ANSWER
        self.embedding    = torch.nn.Embedding(dim_vocab, dim_hidden)
        self.dropout      = torch.nn.Dropout(dropout)
        self.transformer  = torch.nn.Transformer(dim_hidden, num_heads, num_layers,
                                                 num_layers, dim_feedforward,
                                                 dropout=dropout, batch_first=True)
        self.predict      = torch.nn.Linear(dim_hidden, dim_vocab)
        if positional_encoding:
            self.pos_encoding = PositionalEncoding(dim_hidden)
        else:
            self.pos_encoding = torch.nn.Identity()
        ### END ANSWER

    def forward(self, src, tgt):
        ### BEGIN ANSWER
        src_padding_mask = generate_padding_mask(src, self.padding_token)
        tgt_padding_mask = generate_padding_mask(tgt, self.padding_token)
        tgt_mask = generate_square_subsequent_mask(tgt.shape[1], tgt.device)
        src_emb = self.dropout(self.pos_encoding(self.embedding(src)))
        tgt_emb = self.dropout(self.pos_encoding(self.embedding(tgt)))
        out = self.transformer(src_emb, tgt_emb,
                               tgt_mask=tgt_mask,
                               src_key_padding_mask=src_padding_mask,
                               memory_key_padding_mask=src_padding_mask,
                               tgt_key_padding_mask=tgt_padding_mask,
                              )
        return self.predict(out)
        ### END ANSWER


**(e) Try the transformer with an example batch.**

In [ ]:
net = TransformerNetwork(dim_feedforward=72)
x, y = next(iter(train_loader))
y_prev = shift_targets(y)

print('x.shape', x.shape)
print('y.shape', y.shape)
print('y_prev.shape', y_prev.shape)

y_pred = net(x, y_prev)
print('y_pred.shape', y_pred.shape)

# check the shape against what we expected
np.testing.assert_equal(list(y_pred.shape), [y.shape[0], y.shape[1], len(vocab)])

We can convert these predictions to tokens (but they're obviously random):

In [ ]:
print(decode_tokens(torch.argmax(y_pred, dim=2))[:5])

In [ ]:
# Check that the transformer is defined correctly
assert isinstance(net.embedding, torch.nn.Embedding)
assert isinstance(net.dropout, torch.nn.Dropout) # Corrected from net.dropout_layer
assert isinstance(net.transformer, torch.nn.Transformer)
assert isinstance(net.predict, torch.nn.Linear)
# Check parameters of transformer
assert net.transformer.d_model == 64
assert net.transformer.nhead == 4
assert net.transformer.batch_first == True
assert net.transformer.encoder.num_layers == 2
assert net.transformer.decoder.num_layers == 2
assert net.transformer.encoder.layers[0].linear1.out_features == 72
assert net.dropout.p == 0.01 # Corrected from net.dropout_layer.p
assert net.transformer.encoder.layers[0].dropout.p == 0.01
# Check that the forward function behaves correctly
net.train(False)
assert torch.all(torch.isclose( \
            net(x, y_prev), \
            net(torch.cat((x,torch.tensor(vocab['<pad>']).expand(x.shape[0], 5)), axis=1), y_prev), atol=1e-5)), \
       "Adding padding to x should not affect the output of the network. Check src_key_padding_mask and memory_key_padding_mask. The former controls self attention to padding tokens in the encoder, the latter controls cross attention from decoder to encoder."
assert torch.all(torch.isclose( \
            net(x, y_prev), \
            net(x, torch.cat((y_prev,torch.tensor(vocab['<pad>']).expand(y.shape[0], 5)), axis=1))[:,:-5], atol=1e-5)), \
       "Adding padding to y should not affect the output of the network. Check tgt_key_padding_mask."
assert torch.all(torch.isclose( \
            net(x, y_prev)[:,:2], \
            net(x, y_prev[:,:2]), atol=1e-5)), \
       "The presence of later tokens in y should not affect the output for earlier tokens. Check tgt_mask."
assert torch.all(torch.isclose( \
            net(x, y_prev), \
            net(torch.flip(x, [1]), y_prev), atol=1e-5)), \
       "Order of x should not matter for a transformer network. Check src_mask."
assert not torch.all(torch.isclose( \
            net(x, torch.flip(y_prev, [1])), \
            torch.flip(net(x, y_prev), [1]), atol=1e-5)), \
       "Order of y should matter for a transformer network. Check tgt_mask."

## 6.4 Training (10 points)

### Training loop

We will base the training code on last week's code. A complication in computing the loss and accuracy are the padding tokens. So, before we work on the training loop itself, we need to update the `accuracy` function so it ingores these `<pad>` tokens. Let's do this in a generic way

**(a) Copy the `accuracy` function from last week, and add a parameter `ignore_index`. The tokens with `true_y == ignore_index` should be ignored.<span style="float:right"> (1 point)</span>**

Hint: you can select elements from a tensor with `some_tensor[include]` where `include` is a tensor of booleans.

In [ ]:
def accuracy(pred_y, true_y, ignore_index=None):
    # Computes the mean accuracy.
    if pred_y.shape[1] == 1:
        # binary classification
        correct = (pred_y[:, 0] > 0).to(true_y.dtype) == true_y
    else:
        # multi-class classification
        correct = pred_y.argmax(dim=1) == true_y

    # ignore_index mask
    if ignore_index is not None:
      mask = true_y != ignore_index
      correct = correct[mask]

    return int(correct.sum()) / len(correct)

In [ ]:
## Official Solution
def accuracy(pred_y, true_y, ignore_index=None):
### BEGIN ANSWER
    # Computes the mean accuracy.
    # pred_y:       raw network output (before softmax); shape (samples, classes)
    # true_y:       true class labels; shape (samples)
    # ignore_index: (optional) class to ignore
    pred_y = torch.argmax(pred_y, axis=1).to(true_y.dtype)
    correct = (pred_y == true_y).to(torch.float32)
    if ignore_index is not None:
        # Don't look at positions with the ignored token
        correct = correct[true_y != ignore_index]
    return torch.mean(correct)
### END ANSWER

In [ ]:
# Test the accuracy function.
assert accuracy(torch.tensor([[1,0,0],[0.4,0.5,0.1],[0,1,0],[0.4,0.1,0.5]]), torch.tensor([0,1,2,2]), 1) == 2/3
assert accuracy(torch.tensor([[1,0,0],[0.4,0.5,0.1],[0,1,0],[0.4,0.1,0.5]]), torch.tensor([0,1,2,2]), 2) == 1
assert accuracy(torch.tensor([[1,0,0],[0.4,0.5,0.1],[0,1,0],[0.4,0.1,0.5]]), torch.tensor([0,1,2,2]), 3) == 3/4
assert accuracy(torch.tensor([[1,0,0],[0.4,0.5,0.1],[0,1,0],[0.4,0.1,0.5]]), torch.tensor([2,2,1,2]), 2) == 1

**(b) Write a training loop for the transformer model.<span style="float:right"> (4 points)</span>**

See last week's assignment for inspiration.
The code is mostly the same with the following changes:
 * The cross-entropy loss function and accuracy should ignore all `<pad>` tokens. (Use `ignore_index`, see the [documentation of CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html).)
 * The network expects `y_prev` as an extra input.
 * The output of the network contains a batch of N samples, with maximum length L, and gives logits over C classes, so it has size (N,L,C). But `CrossEntropyLoss` and `accuracy` expect a tensor of size (N,C,L). You can use [torch.Tensor.transpose](https://pytorch.org/docs/stable/generated/torch.transpose.html) to change the output to the right shape.

In [ ]:
class Metrics:
    """Accumulate mean values of one or more metrics."""
    def __init__(self, n):
        self.count = 0
        self.sum = (0,) * n
    def add(self, count, *values):
        self.count += count
        self.sum = tuple(s + count * v for s,v in zip(self.sum,values))
    def mean(self):
        return tuple(s / self.count for s in self.sum)

def evaluate(net, test_loader, loss_function=torch.nn.CrossEntropyLoss(), device=device):
    """
    Evaluate a model on the given dataset.
    Return loss, accuracy
    """
    with torch.no_grad():
        net.eval()
        metrics = Metrics(2)
        for x, y in test_loader:
            x = x.to(device)
            y = y.to(device)
            # Shift targets to create y_prev for the decoder input
            y_prev = shift_targets(y)
            y_prev = y_prev.to(device)

            pred_y = net(x, y_prev)

            # Transpose for CrossEntropyLoss: (N, L, C) -> (N, C, L)
            pred_y_reshaped = pred_y.transpose(1, 2)
            loss = loss_function(pred_y_reshaped, y)
            acc = accuracy(pred_y_reshaped, y, ignore_index=net.padding_token)
            metrics.add(len(y), loss.item(), acc)
        return metrics.mean()


class Plotter:
    """For plotting data in animation."""
    # Based on d2l.Animator
    def __init__(self, xlabel=None, ylabel=None, legend=None, xlim=None,
                  ylim=None, xscale='linear', yscale='linear',
                  titles=[],
                  fmts=('-', '--', '-.', ':'), nrows=1, ncols=1,
                  figsize=(5, 3)):
        # Incrementally plot multiple lines
        if legend is None:
            legend = []
        plt.style.use('ggplot')
        self.fig, self.axes = plt.subplots(nrows, ncols, figsize=(figsize[0] * ncols, figsize[1] * nrows))
        if nrows * ncols == 1:
            self.axes = [self.axes, ]
        # Use a function to capture arguments
        def config_axes():
            for axis, title in zip(self.axes, titles):
                axis.set_xlabel(xlabel), axis.set_ylabel(ylabel)
                axis.set_xscale(xscale), axis.set_yscale(yscale)
                axis.set_xlim(xlim),     axis.set_ylim(ylim)
                axis.set_title(title)
                if legend:
                    axis.legend(legend)
        self.config_axes = config_axes
        self.legend = legend
        self.data = [{leg: dict(x=[], y=[], fmt=fmt) for leg,fmt in zip(legend,fmts)} for _ in range(ncols)]

    def add(self, line, x, y):
        if not hasattr(y, "__len__"):
            y = [y]
        if not hasattr(x, "__len__"):
            x = [x] * len(y)
        for a, b, data in zip(x, y, self.data):
            if a is not None and b is not None:
                data[line]['x'].append(a)
                data[line]['y'].append(b)
        self.show()

    def show(self):
        for axis, data in zip(self.axes, self.data):
            axis.cla()
            for line in self.legend:
                line_data = data[line]
                axis.plot(line_data['x'], line_data['y'], line_data['fmt'])
        self.config_axes()
        display.display(self.fig)
        display.clear_output(wait=True)


In [ ]:
def train(net, data_loaders, num_epochs, lr=0.001, optimizer=torch.optim.Adam, device=device):
    """
    Train a network on the given data set.
    After every epoch compute validation loss and accuracy.
    """
    net.to(device)
    train_loader = data_loaders['train']
    num_batches = len(train_loader)
    optimizer = optimizer(net.parameters(), lr=lr)
    loss_function = torch.nn.CrossEntropyLoss(ignore_index=net.padding_token)
    plotter = Plotter(xlabel='epoch', xlim=[1, num_epochs], ncols=2,titles=['loss','accuracy'], legend=['train','validation'])
    start_time = time.time()
    for epoch in range(num_epochs):
        # Sum of training loss, sum of training accuracy, no. of examples
        net.train()
        metrics = Metrics(2)
        for i, (x, y) in enumerate(train_loader):
            optimizer.zero_grad()
            x = x.to(device)
            y = y.to(device)

            y_prev = shift_targets(y)
            y_prev = y_prev.to(device)

            pred_y = net(x, y_prev)

            # Transpose for CrossEntropyLos
            pred_y = pred_y.transpose(1, 2)
            loss = loss_function(pred_y, y)
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                acc = accuracy(pred_y, y, ignore_index=net.padding_token)
                metrics.add(len(y), loss.item(), acc)
            if (i + 1) % (num_batches // 5) == 0 or i == num_batches - 1:
                train_loss, train_acc = metrics.mean()
                plotter.add('train', epoch + (i + 1) / num_batches, (train_loss, train_acc))

        val_loss, val_acc = evaluate(net, data_loaders['val'], loss_function=loss_function, device=device)
        plotter.add('validation', epoch + 1, (val_loss, val_acc))

    train_loss, train_acc = metrics.mean()
    train_time = time.time() - start_time
    print(f'train loss {train_loss:.3f}, train acc {train_acc:.3f}, '
          f'val loss {val_loss:.3f}, val acc {val_acc:.3f}')
    print(f'{metrics.count * num_epochs / train_time:.1f} samples/sec '
          f'on {str(device)}')

6.4 b
3.5
Add comment
(b) Write a training loop for the transformer model.(4 points)

See last week's assignment for inspiration. The code is mostly the same with the following changes:

The cross-entropy loss function and accuracy should ignore all <pad> tokens. (Use ignore_index, see the documentation of CrossEntropyLoss.)
The network expects y_prev as an extra input.
The output of the network contains a batch of N samples, with maximum length L, and gives logits over C classes, so it has size (N,L,C). But CrossEntropyLoss and accuracy expect a tensor of size (N,C,L). You can use torch.Tensor.transpose to change the output to the right shape.
Grading:

1 point for a working training loop.
1 point for correct use of ignore_index in loss and accuracy.
1 point for building y_prev and passing it to net.
1 point for correct use of pred_y.transpose(1,2).
Note: your train function might differ in terms of ploting and printing metrics.

In [ ]:
## Correct Solution
### BEGIN ANSWER
class Plotter:
    """For plotting data in animation."""
    # Based on d2l.Animator
    def __init__(self, xlabel=None, ylabel=None, legend=None, xlim=None,
                 ylim=None, xscale='linear', yscale='linear',
                 titles=[],
                 fmts=('-', '--', '-.', ':'), nrows=1, ncols=1,
                 figsize=(5, 3)):
        # Incrementally plot multiple lines
        if legend is None:
            legend = []
        plt.style.use('ggplot')
        self.fig, self.axes = plt.subplots(nrows, ncols, figsize=(figsize[0] * ncols, figsize[1] * nrows))
        if nrows * ncols == 1:
            self.axes = [self.axes, ]
        # Use a function to capture arguments
        def config_axes():
            for axis, title in zip(self.axes, titles):
                axis.set_xlabel(xlabel), axis.set_ylabel(ylabel)
                axis.set_xscale(xscale), axis.set_yscale(yscale)
                axis.set_xlim(xlim),     axis.set_ylim(ylim)
                axis.set_title(title)
                if legend:
                    axis.legend(legend)
        self.config_axes = config_axes
        self.legend = legend
        self.data = [{leg: dict(x=[], y=[], fmt=fmt) for leg,fmt in zip(legend,fmts)} for _ in range(ncols)]

    def add(self, line, x, y):
        if not hasattr(y, "__len__"):
            y = [y]
        if not hasattr(x, "__len__"):
            x = [x] * len(y)
        for a, b, data in zip(x, y, self.data):
            if a is not None and b is not None:
                data[line]['x'].append(a)
                data[line]['y'].append(b)
        self.show()

    def show(self):
        for axis, data in zip(self.axes, self.data):
            axis.cla()
            for line in self.legend:
                line_data = data[line]
                axis.plot(line_data['x'], line_data['y'], line_data['fmt'])
        self.config_axes()
        display.display(self.fig)
        display.clear_output(wait=True)
### END ANSWER


### BEGIN ANSWER
class Metrics:
    """Accumulate mean values of one or more metrics."""
    def __init__(self, n):
        self.count = 0
        self.sum = (0,) * n
    def add(self, count, *values):
        self.count += count
        self.sum = tuple(s + count * v for s,v in zip(self.sum,values))
    def mean(self):
        return tuple(s / self.count for s in self.sum)

def evaluate(net, test_loader, loss_function, device=device):
    """
    Evaluate a model on the given dataset.
    Return loss, accuracy
    """
    padding_token = vocab['<pad>']
    with torch.no_grad():
        net.eval()
        metrics = Metrics(2)
        for x, y in test_loader:
            x = x.to(device)
            y = y.to(device)
            # predict next token based on previous tokens
            y_prev = shift_targets(y)
            pred_y = net(x, y_prev)
            # Note: network has classes as last dim, CrossEntropyLoss expects classes as dim=1
            pred_y = pred_y.transpose(1,2)
            loss = loss_function(pred_y, y)
            acc = accuracy(pred_y, y, ignore_index=padding_token)
            metrics.add(len(y), loss.item(), acc)
        return metrics.mean()
### END ANSWER



def train(net, data_loaders, num_epochs=100, lr=0.001, optimizer=torch.optim.Adam, device=device):
    """
    Train a network on the given data set.
    After every epoch compute validation loss and accuracy.
    """
    ### BEGIN ANSWER
    net.to(device)
    train_loader = data_loaders['train']
    num_batches = len(train_loader)
    optimizer = optimizer(net.parameters(), lr=lr)
    padding_token = vocab['<pad>']
    bos_token = vocab['<bos>']
    loss_function = torch.nn.CrossEntropyLoss(ignore_index=padding_token)

    plotter = Plotter(xlabel='epoch', xlim=[1, num_epochs], ncols=2,
                      titles=['loss','accuracy'], legend=['train','validation'])
    start_time = time.time()
    for epoch in range(num_epochs):
        # Sum of training loss, sum of training accuracy, no. of examples
        net.train()
        metrics = Metrics(2)
        for i, (x, y) in enumerate(train_loader):
            optimizer.zero_grad()
            x = x.to(device)
            y = y.to(device)
            # predict next token based on true previous token
            y_prev = shift_targets(y, bos_token)
            pred_y = net(x, y_prev)
            # Note: network has classes as last dim, CrossEntropyLoss expects classes as dim=1
            pred_y = pred_y.transpose(1,2)
            # optimizer step
            loss = loss_function(pred_y, y)
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                acc = accuracy(pred_y, y, ignore_index=padding_token)
                metrics.add(len(y), loss.item(), acc)

        # Update plotter
        train_loss, train_acc = metrics.mean()
        val_loss, val_acc = evaluate(net, data_loaders['val'], loss_function=loss_function, device=device)
        plotter.add('train', epoch + (i + 1) / num_batches, (train_loss, train_acc))
        plotter.add('validation', epoch + 1, (val_loss, val_acc))

    train_loss, train_acc = metrics.mean()
    train_time = time.time() - start_time
    print(f'train loss {train_loss:.3f}, train acc {train_acc:.3f}, '
          f'val loss {val_loss:.3f}, val acc {val_acc:.3f}')
    print(f'{metrics.count * num_epochs / train_time:.1f} samples/sec '
          f'on {str(device)}')
    ### END ANSWER


### Experiment

**(c) Train a transformer network. Use 100 epochs with a learning rate of 0.001<span style="float:right"> (no points)</span>**

In [ ]:
# TODO: your answer here
net = TransformerNetwork()
train(net, data_loaders, num_epochs=100, lr=0.001)

**(d) Briefly discuss the results. Has the training converged? Is this a good calculator?<span style="float:right"> (1 point)</span>**

No, training has not converged, but the network is overfitting. The validation accuracy is low. It is not a good calculator.

**(e) Run the trained network with input `"123+123"` and `"321+321"`.<span style="float:right"> (1 point)</span>**

In [ ]:
def predict(net, q, a):
    # Run net to predict the output given the input `q` and y_prev based on `a`.
    # Return predicted y
    with torch.no_grad():
        # tokenize input
        q_tokens_list = tokenize_and_encode(q)
        a_tokens_list = tokenize_and_encode(a)

        # Convert lists to tensors and add batch dimension
        q_tensor = torch.tensor(q_tokens_list, device=device).unsqueeze(0)
        a_tensor = torch.tensor(a_tokens_list, device=device).unsqueeze(0)

        # predict y
        y_pred = net(q_tensor, a_tensor)

    return y_pred


for src, tgt in [('123+123', '246'), ('321+321', '642')]:
    print(f'For {src}={tgt}')
    y_pred = predict(net, src, tgt)
    print('  y_pred[0]', y_pred[0])
    print('  encoded', torch.argmax(y_pred, dim=-1))
    print('  tokens', decode_tokens(torch.argmax(y_pred, dim=-1)))
    print()

In [ ]:
## OFFICIAL SOLUTION
def predict(net, q, a):
    # Run net to predict the output given the input `q` and y_prev based on `a`.
    # Return predicted y
    with torch.no_grad():
        ### BEGIN ANSWER
        x = torch.tensor(tokenize_and_encode(q), device=device)[None, :]
        y = torch.tensor(tokenize_and_encode(a), device=device)[None, :]
        y_prev = shift_targets(y)
        return net(x, y_prev)[0]
        ### END ANSWER

for src, tgt in [('123+123', '246'), ('321+321', '642')]:
    print(f'For {src}={tgt}')
    y_pred = predict(net, src, tgt)
    print('  y_pred[0]', y_pred[0])
    print('  encoded', torch.argmax(y_pred, dim=-1))
    print('  tokens', decode_tokens(torch.argmax(y_pred, dim=-1)))
    print()




**(f) Compare the predictions for the first element of y for the two different inputs. Can you explain why these are the same for different inputs?<span style="float:right"> (1 point)</span>**

TODO: Your answer here.


**OFFICIAL SOLUTION**:  
The predictions for the first element of `y` are identical. When not using positional encoding, all tokens are treated equally, so the encoder's output is effectively the same for the two questions (only in a different order). The first outputs for the decoder will therefore also be the same. Later outputs can differ, because `y_prev` will be different.

**(g) Does the validation accuracy estimate how often the model is able to answers formulas correctly? Explain your answer.<span style="float:right"> (1 point)</span>**

TODO: Your answer here.

**CORRECT SOLUTION:**  
The validation accuracy is measured per token. An error rate of 20% per token would still result in around a 50% error rate for the whole answer. It would be better to measure the accuracy for the entire token sequence. Still, a very high accuracy does put a lower bound on the whole answer accuracy.

**(h) If the forward function takes the shifted output `y_prev` as input, how can we use it if we don't know the output yet?<span style="float:right"> (1 point)</span>**

TODO: Your answer here.

**OFFICIAL SOLUTION:**  
The prediction must be done one token at a time. The predicted token becomes the input `y_prev`for predicting the next token.

## 6.5 Positional encoding (5 points)

We did not yet include positional encoding in the network.
PyTorch does not include such an encoder. So, here we define such a module ourselves:

In [ ]:
class PositionalEncoding(nn.Module):
    """Positional encoding."""
    def __init__(self, num_hiddens, max_len=100):
        super().__init__()
        # Create a long enough matrix of position encodings P
        positions = torch.arange(max_len, dtype=torch.float32)
        freqs = torch.pow(max_len, (1 + 2 * torch.arange(num_hiddens / 2)) / num_hiddens)
        X = positions[:,None] / freqs[None,:]
        self.P = torch.zeros((max_len, num_hiddens))
        self.P[:, 0::2] = torch.sin(X)
        self.P[:, 1::2] = torch.cos(X)

    def forward(self, X):
        return X + self.P[None, :X.shape[1], :].to(X.device)

**(a) Add positional encoding to the TransformerModel.<span style="float:right"> (point given in earlier question)</span>**

In [ ]:
# TODO: Modify the code in 6.3d.

class TransformerPosNetwork(torch.nn.Module):
    def __init__(self,
                 dim_vocab=len(vocab), padding_token=vocab['<pad>'],
                 num_layers=2, num_heads=4, dim_hidden=64, dim_feedforward=64,
                 dropout=0.01, positional_encoding=False):
        super().__init__()
        self.padding_token = padding_token
        self.embedding    = torch.nn.Embedding(num_embeddings=dim_vocab, embedding_dim=dim_hidden)
        self.dropout      = torch.nn.Dropout(p=dropout)
        self.transformer  = torch.nn.Transformer(
            d_model=dim_hidden,
            nhead=num_heads,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.predict      = torch.nn.Linear(dim_hidden, dim_vocab)
        self.positional_encoding_enabled = positional_encoding
        if positional_encoding:
            self.pos_encoding = PositionalEncoding(num_hiddens=dim_hidden, max_len=max_len)
        else:
            self.pos_encoding = torch.nn.Identity()

    def forward(self, src, tgt):
        # Apply embedding
        src_emb = self.dropout(self.embedding(src))
        tgt_emb = self.dropout(self.embedding(tgt))

        # with positional encoding
        if self.pos_encoding:
            src_emb = self.pos_encoding(src_emb)
            tgt_emb = self.pos_encoding(tgt_emb)

        # create masks
        src_key_padding_mask = generate_padding_mask(src, self.padding_token)
        tgt_key_padding_mask = generate_padding_mask(tgt, self.padding_token)
        memory_key_padding_mask = src_key_padding_mask # Encoder output padding mask for decoder attention

        # square subsequent mask
        tgt_mask = generate_square_subsequent_mask(tgt.shape[1])

        # create output
        output = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            src_mask=None,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )

        # prediction
        output = self.predict(output)
        return output



**(b) Construct and train a network with positional encoding<span style="float:right"> (1 point)</span>**

In [ ]:
# TODO: your answer here
net_pos = TransformerPosNetwork()

train(net_pos, data_loaders, num_epochs=100, lr=0.001)

**(c) How does the performance of a model with positional encoding compare to a model without?<span style="float:right"> (1 point)</span>**

TODO: Your answer here.

**(d) Run the trained network with input `"123+123"` and `"321+321"`.<span style="float:right"> (no points)</span>**

In [ ]:
# TODO: Your code here.

**(e) Compare the predictions for the first element of y with what you found earlier. Can you explain what happens?<span style="float:right"> (1 point)</span>**

TODO: Your answer here.

**(f) Explain in your own words why positional encoding is used in transformer networks.<span style="float:right"> (1 point)</span>**

TODO: Your answer here.

**(g) Look at the learning curve. Can you suggest a way to improve the performance of the model?<span style="float:right"> (1 point)</span>**

TODO: Your answer here.

**(h) Optional: if time permits, try to train an even better model**

## 6.6 Predicting for new samples (5 points)

Predicting an output given a new sample requires an appropriate search algorithm (see [d2l chapter 10.8](https://d2l.ai/chapter_recurrent-modern/beam-search.html)). Here, we will implement the simplest form: a greedy search algorithm that selects the token with the highest probability at each time step.

**(a) Describe this search strategy in pseudo-code.<span style="float:right"> (1 point)</span>**

TODO: Your answer here.

**(b) Implement a greedy search function to predict a sequence using `net_pos`.<span style="float:right"> (2 points)</span>**

In [ ]:
def predict_greedy(net, src, length):
    # predict an output sequence of the given (maximum) length given input string src
    with torch.no_grad():
        # TODO: Your code here.
        pass

predicted_sequence = predict_greedy(net_pos, '123+123', 6)
print(decode_tokens(predicted_sequence))

**(c) Does this search strategy give a high-quality prediction? Why, or why not?<span style="float:right"> (1 point)</span>**

TODO: Your answer here.

**(d) What alternative search strategy could we use to improve the predictions? Why would this help?<span style="float:right"> (1 point)</span>**

TODO: Your answer here.

## 6.7 Discussion (4 points)

Last week, we looked at recurrent neural networks such as the LSTM. Both recurrent neural networks and transformers work with sequences, but in recent years the transformer has become more popular than the recurrent models.

**(a) An advantage of transformers over recurrent neural networks is that they can be faster to train. Why is that?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
That is because of the ability of transformers to process context in parallel. The input to the transformer is given all at once, but for RNNs the input is given one token at the time. This recursive structure cannot be trained in parallel.
Whereas RNNs perform sequential processes, meaning sequences must be processed token by token.

Inference time RNN: $O(N)$  
Inference time Transformer: $O(N^2)$

**OFFICIAL SOLUTION:**  
Transformers can train in parallel for all tokens, whereas RNNs are inherently sequential.

**(b) Does this advantage also hold when predicting outputs for new sequences? Why, or why not?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
No, this advantage doesn't carry over to the inference step, because the nature of prediction is still sequential in nature, aka it depends on tokens that were generated before.


**OFFICIAL SOLUTION**:  
No, predicting tokens is still sequential, because the previous predicted token is given as an input to the decodeer in the form of `y_prev`.

**(c) Why is positional encoding often used in transformers, but not in convolutional or recurrent neural networks?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
Conv Nets and RNNs don't need positional encoding of tokens, because they already encode positional information in the form of receptive fields and hidden states.

**OFFICIAL SOLUTION:**  
RNNs do not need positional encoding, because the position of the tokens can be inferred from the order in which they are given to the network. In a CNN, ususally the input is assumed to be translational invariant or equivariant, and the network is designed to have this invariance. Adding positional encoding would only defeat that.

The structure of a recurrent neural network makes it very suitable for online predictions, such as real-time translation, because it only depends on prior inputs. You can design an architecture where the RNN produces an output token for every input token given to it, and it can produce that output without having to wait for the rest of the input.

Note: 'online' means producing outputs continuously as new input comes in, as opposed to collecting a full dataset and analyzing it afterwards, it has nothing to do with the internet.

**(d) How would a transformer work in an online application? Do you need to change the architecture?<span style="float:right"> (1 point)</span>**

**ANSWER:**  

Pass


**CORRECT SOLUTION:**  
YOu would add more inputs over time. You could use masked self attention alsos for the encoder, so that the new input tokens can only attend to earlier tokens. Then new inputs can be added without having to recompute everything.

A potential issue is the $O(n^2)$ complexity of self attention blocks. RNNs do not have this issue and they can operate with a fixed memory size and computation budget.

## The end

Well done! Please double check the instructions at the top before you submit your results.

*This assignment has 47 points.*
<span style="float:right;color:#aaa;font-size:10px;"> Version 5a33150 / 2025-12-09</span>